# Saccadion-1: One-Click Colab Free TPU Data Downloader & Smart Epoch-by-Epoch Trainer

**Before running:** Select `Runtime -> Change runtime type -> TPU (v2-8 or v5e-1)`.

This notebook automatically:
1. Clones `https://github.com/vash-001/Saccadion.git` and mounts Google Drive for persistent checkpoints.
2. Downloads & organizes high-Elo games (**Lichess Elite 2500+** & **Official Stockfish Fishtest 3600+**) with automatic **10-20 ply Strategic Pressure** targets.
3. Trains **Saccadion-1** epoch-by-epoch with **JAX + Muon + Meta-Dopamine** showing ALL losses, Top-1/Top-3 move accuracy, WDL accuracy, Centipawn MAE, APL 92% sparsity, and a 6-panel live chart.
4. Exports `saccadion_best.saccnet` directly for your 100% Pure-Rust CPU engine.

In [ ]:
# Step 1: Clone repository, mount Drive, and install lightweight dependencies
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/vash-001/Saccadion.git /content/Saccadion || (cd /content/Saccadion && git pull --ff-only)
!pip install -q optax python-chess zstandard matplotlib

import jax
print('JAX version:', jax.__version__, '| TPU Devices:', jax.devices())

In [ ]:
# Step 2 (Optional but 100x faster): Build Pure-Rust saccadion-data shard packer on Colab
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
!export PATH="$HOME/.cargo/bin:$PATH" && cd /content/Saccadion && cargo build --release -p saccadion-data

In [ ]:
# Step 3: Download & Organize Chess Datasets (Lichess Elite 2500+ & Stockfish Fishtest 3600+)
import sys
from pathlib import Path
sys.path.insert(0, '/content/Saccadion/training')

from saccadion_tpu.fetch_and_prepare_data import fetch_and_prepare

PERSIST = Path('/content/drive/MyDrive/Saccadion_TPU')
SHARD_DIR = fetch_and_prepare(
    out_dir=PERSIST / 'data',
    source='auto',       # 'auto' | 'lichess_elite' | 'fishtest' | 'lichess_eval'
    max_games=4000,      # Increase to 25000+ for Pro training
    val_pct=5,
)
print('Ready shards in:', SHARD_DIR)

In [ ]:
# Step 4: Smart Epoch-by-Epoch TPU Training with Full Telemetry & Auto-Best .saccnet Export
from saccadion_tpu.config import SaccadionConfig
from saccadion_tpu.train import train_saccadion_smart
from IPython.display import Image, display

TIER = 'nano'  # 'tiny' | 'nano' | 'pro'
cfg = {'tiny': SaccadionConfig.tiny, 'nano': SaccadionConfig.nano, 'pro': SaccadionConfig.pro}[TIER]()

CKPT_DIR = PERSIST / f'checkpoints_{TIER}'
EXPORT_DIR = PERSIST / f'exports_{TIER}'

params, history = train_saccadion_smart(
    cfg=cfg,
    shard_dir=SHARD_DIR,
    checkpoint_dir=CKPT_DIR,
    export_dir=EXPORT_DIR,
    epochs=15,
    batch_size=1024,
    lr_muon=1.5e-2,
    lr_adamw=1.5e-3,
)

display(Image(filename=str(CKPT_DIR / 'training_dashboard.png')))
print('Best Pure-Rust weights ready at:', EXPORT_DIR / 'saccadion_best.saccnet')